# OpenRouter API — nvidia/nemotron-3-super-120b

[OpenRouter](https://openrouter.ai/) provides a unified API for 100+ language models.
This notebook demonstrates calling the **nvidia/nemotron-3-super-120b-a12b:free** model.

**Setup:**
1. Create a free account at https://openrouter.ai/
2. Generate an API key at https://openrouter.ai/keys
3. Set `OPENROUTER_API_KEY` below (or as an environment variable)

**Requirements:** `pip install requests`

In [ ]:
import requests
import os
import json

## Configuration

In [ ]:
# Set your OpenRouter API key here or via environment variable OPENROUTER_API_KEY
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "YOUR_OPENROUTER_KEY")

MODEL = "nvidia/nemotron-3-super-120b-a12b:free"  # remove ':free' for paid tier

API_URL = "https://openrouter.ai/api/v1/chat/completions"

## Chat completion helper

In [ ]:
def chat(content, model=MODEL, system=None, temperature=0.7, max_tokens=1024):
    """
    Send a message to the OpenRouter API and return the assistant reply.

    Args:
        content     : User message string
        model       : OpenRouter model ID
        system      : Optional system prompt string
        temperature : Sampling temperature (0.0 – 1.0)
        max_tokens  : Maximum tokens to generate

    Returns:
        str: The assistant's reply text
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": content})

    response = requests.post(
        API_URL,
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": model,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
        },
        timeout=120
    )

    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]

## Basic usage

In [ ]:
reply = chat("Explain the difference between supervised and unsupervised learning in two sentences.")
print(reply)

## With a system prompt

In [ ]:
reply = chat(
    content="What is PyTorch and why is it popular for deep learning?",
    system="You are a concise technical writer. Answer in bullet points."
)
print(reply)

## Raw request (matches OpenRouter docs exactly)

In [ ]:
CONTENT = "Write a haiku about neural networks."

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    },
    json={
        "model": "nvidia/nemotron-3-super-120b-a12b:free",  # or without :free for paid
        "messages": [{"role": "user", "content": CONTENT}]
    }
)

data = response.json()
print(json.dumps(data, indent=2))

## Multi-turn conversation

In [ ]:
def multi_turn_chat(turns, model=MODEL):
    """
    Send a full conversation history to the API.
    
    Args:
        turns: list of {"role": "user"|"assistant"|"system", "content": "..."}
    """
    response = requests.post(
        API_URL,
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        },
        json={"model": model, "messages": turns},
        timeout=120
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


conversation = [
    {"role": "user",      "content": "What is gradient descent?"},
    {"role": "assistant", "content": "Gradient descent is an optimisation algorithm that iteratively adjusts model parameters in the direction of steepest loss decrease."},
    {"role": "user",      "content": "What are its main variants?"},
]

reply = multi_turn_chat(conversation)
print(reply)

## List available models on OpenRouter

In [ ]:
models_resp = requests.get(
    "https://openrouter.ai/api/v1/models",
    headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
)
models = models_resp.json().get("data", [])

# Filter free models
free_models = [m for m in models if ":free" in m["id"]]
print(f"Free models available: {len(free_models)}")
for m in free_models[:20]:
    print(f"  {m['id']}")